# ML_U4_Lab02 — Clustering Jerárquico en Práctica

**Versión:** 2025-1 | **Modificado:** 2026-05-30
**Dataset:** Iris + sintéticos | **Duración:** 1 hora
**Modalidad:** Individual o parejas

---

## 📋 Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| Setup | Imports y datos | 5 min | Todos |
| PARTE 1 | Sin computador: análisis de dendrogramas | 10 min | Todos |
| PARTE 2 | Dendrogramas y criterios de enlace | 20 min | Todos |
| PARTE 3 | Selección de K y comparación con K-means | 20 min | Todos |
| ANÁLISIS | Interpretación y reflexión | 5 min | Diferenciado |

---

## 🎯 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|----------|
| Obligatorio | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| Entrega | .ipynb ejecutado | .ipynb ejecutado |

## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_iris_sc = StandardScaler().fit_transform(X_iris)

# Dataset con estructura clara para visualización
X_vis, y_vis = make_blobs(n_samples=80, centers=3, cluster_std=0.6, random_state=RANDOM_STATE)
X_vis_sc = StandardScaler().fit_transform(X_vis)

import sklearn, scipy
print(f"✅ Setup completo | sklearn {sklearn.__version__} | scipy {scipy.__version__}")
print(f"   Iris: {X_iris.shape} | Dataset visualización: {X_vis.shape}")

---
## PARTE 1 — Sin Computador: Dendrogramas (10 min) 🖊️

---
## PARTE 2 — Dendrogramas y Criterios de Enlace (20 min)

In [ ]:
# ━━━ TODO 1: GENERAR Y COMPARAR DENDROGRAMAS ━━━
# Para el dataset X_vis_sc (3 clusters bien separados):
#
# 1. Calcula la matriz de linkage para los 4 métodos: 'single', 'complete', 'average', 'ward'
#    usando scipy.cluster.hierarchy.linkage(X, method=method)
#
# 2. Grafica los 4 dendrogramas en una figura 2x2
#    usa dendrogram(Z, truncate_mode='lastp', p=15, no_labels=True)
#    añade una línea horizontal con la altura de corte sugerida para K=3
#
# 3. Para cada método, agrega el título con:
#    - Nombre del método
#    - Altura máxima de fusión (Z[:, 2].max())

# ESCRIBE TU CÓDIGO AQUÍ:
methods = ['single', 'complete', 'average', 'ward']
Z_dict = {}  # guarda los linkage matrices

# for method in methods:
#     Z_dict[method] = linkage(X_vis_sc, method=method)

# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# ...

if len(Z_dict) > 0:
    print(f"✅ Linkage matrices calculadas para: {list(Z_dict.keys())}")
else:
    print("❌ TODO 1 sin implementar")

In [ ]:
# ━━━ TODO 2: CORRELACIÓN COFENÉTICA ━━━
# Para cada método en Z_dict:
#   1. Calcula la correlación cofenética con scipy.cluster.hierarchy.cophenet(Z, pdist(X))
#      (retorna (c, coph_dists) — usa solo c)
#   2. Imprime una tabla: Método | Corr. Cofenética | Altura máxima fusión
#   3. ¿Qué método tiene mayor correlación cofenética?

# ESCRIBE TU CÓDIGO AQUÍ:
cophenetic_results = {}  # {method: corr_cofenética}

# for method, Z in Z_dict.items():
#     c, _ = cophenet(Z, pdist(X_vis_sc))
#     cophenetic_results[method] = c

if len(cophenetic_results) > 0:
    print(f"{'Método':<15} {'Corr. Cofenética':>18}")
    for m, c in cophenetic_results.items():
        print(f"{m:<15} {c:>18.4f}")
else:
    print("❌ TODO 2 sin implementar")

In [ ]:
# 🔍 Tests de sanidad — Parte 2 (NO MODIFICAR)
try:
    assert len(Z_dict) == 4, f"Se esperan 4 métodos, se encontraron {len(Z_dict)}"
    for method, Z in Z_dict.items():
        assert Z.shape[1] == 4, f"{method}: la matriz linkage debe tener 4 columnas"
        assert Z.shape[0] == len(X_vis_sc) - 1, f"{method}: deben haber n-1 fusiones"
    print(f"✅ PASS — Linkage matrices: {len(Z_dict)} métodos, shapes correctas")
except (AssertionError, AttributeError) as e:
    print(f"❌ FAIL — {e}")

try:
    assert len(cophenetic_results) == 4, "Faltan correlaciones cofenéticas"
    assert all(0.5 <= c <= 1.0 for c in cophenetic_results.values()), \
        "Correlaciones fuera del rango esperado [0.5, 1.0]"
    best_method = max(cophenetic_results, key=cophenetic_results.get)
    print(f"✅ PASS — Mejor correlación cofenética: {best_method} ({cophenetic_results[best_method]:.4f})")
except (AssertionError, ValueError) as e:
    print(f"❌ FAIL — {e}")

---
## PARTE 3 — Selección de K y Comparación con K-means (20 min)

In [ ]:
# ━━━ TODO 3: SELECCIÓN DE K DESDE EL DENDROGRAMA DE IRIS ━━━
# Usando Ward sobre Iris (X_iris_sc):
#
# 1. Calcula el linkage Ward: Z_iris = linkage(X_iris_sc, method='ward')
# 2. Grafica el dendrograma (truncado a los últimos 30 nodos)
# 3. Identifica el K más natural mirando la brecha más grande entre
#    distancias de fusión consecutivas (Z_iris[-30:, 2])
# 4. Marca ese corte en el dendrograma con una línea horizontal
# 5. Extrae los labels con fcluster(Z_iris, K_natural, criterion='maxclust')
# 6. Calcula ARI con y_iris

# ESCRIBE TU CÓDIGO AQUÍ:
Z_iris_ward = None  # reemplaza con linkage(X_iris_sc, method='ward')
K_natural = None    # el K que identificaste
labels_ward = None  # fcluster resultado
ari_ward = None     # ARI

In [ ]:
# ━━━ TODO 4: TABLA COMPARATIVA — 4 MÉTODOS vs K-MEANS (K=3 EN IRIS) ━━━
# Evalúa todos los métodos con K=3 en Iris y crea una tabla comparativa.
#
# Columnas: Método | Silhouette | ARI vs. verdad real
# Filas: K-means, single, complete, average, ward
#
# Pista: para los métodos jerárquicos, usa AgglomerativeClustering(n_clusters=3, linkage=method)
#        El ARI se calcula contra y_iris (las especies reales)

# ESCRIBE TU CÓDIGO AQUÍ:
comparison_results = {}  # {nombre: {'silhouette': ..., 'ari': ...}}

# for name, model in [('K-means', KMeans(n_clusters=3, ...)), ...]:
#     labels = model.fit_predict(X_iris_sc)
#     comparison_results[name] = {
#         'silhouette': silhouette_score(X_iris_sc, labels),
#         'ari': adjusted_rand_score(y_iris, labels)
#     }

if len(comparison_results) > 0:
    print(f"{'Método':<25} {'Silhouette':>12} {'ARI':>8}")
    print("-" * 50)
    for name, metrics in comparison_results.items():
        print(f"{name:<25} {metrics['silhouette']:>12.4f} {metrics['ari']:>8.4f}")
else:
    print("❌ TODO 4 sin implementar")

In [ ]:
# 🔍 Tests de sanidad — Parte 3 (NO MODIFICAR)
try:
    assert Z_iris_ward is not None, "TODO 3: linkage Ward no calculado"
    assert Z_iris_ward.shape[0] == len(X_iris_sc) - 1, "Linkage shape incorrecto"
    print(f"✅ PASS — Linkage Ward Iris: {Z_iris_ward.shape[0]} fusiones")
except (AssertionError, TypeError) as e:
    print(f"❌ FAIL — {e}")

try:
    assert labels_ward is not None, "TODO 3: labels Ward no calculados"
    assert ari_ward is not None and 0.0 <= ari_ward <= 1.0, "ARI fuera de rango"
    print(f"✅ PASS — Ward K={K_natural}: ARI={ari_ward:.4f}")
except (AssertionError, TypeError) as e:
    print(f"❌ FAIL — {e}")

try:
    assert len(comparison_results) >= 5, "Se esperan al menos 5 métodos"
    for name, m in comparison_results.items():
        assert 0 <= m['silhouette'] <= 1 and 0 <= m['ari'] <= 1
    print(f"✅ PASS — Tabla comparativa: {len(comparison_results)} métodos")
except (AssertionError, KeyError, TypeError) as e:
    print(f"❌ FAIL — {e}")

---
## ✅ Checklist de Entrega

### Pregrado
- [ ] Parte 1: preguntas 1–4 respondidas a mano
- [ ] TODO 1: 4 dendrogramas graficados y comparados
- [ ] TODO 2: correlaciones cofenéticas calculadas
- [ ] TODO 3: K natural identificado desde dendrograma de Iris
- [ ] TODO 4: tabla comparativa de 5 métodos
- [ ] Preguntas de análisis 1–3 respondidas

### Doctorado (adicional)
- [ ] Parte 1: preguntas 5–6 respondidas
- [ ] TODO [PhD]: Ward + K-means combinado implementado
- [ ] Preguntas P4–P5 respondidas